# ReviewLens AI — Model Training
**Run this notebook on Google Colab with a T4 GPU (free)**

This notebook will:
1. Install dependencies
2. Download the Amazon Reviews dataset from Kaggle
3. Run the data pipeline
4. Fine-tune DistilBERT for sentiment analysis
5. Fine-tune T5-small for summarization
6. Package both models into a ZIP for download

**Estimated time: ~25-35 minutes on T4 GPU**

---
### Before running:
1. Go to **Runtime → Change runtime type → T4 GPU**
2. Upload your `kaggle.json` file when asked in Cell 2

In [ ]:
# ── Cell 1: Check GPU ─────────────────────────────────────────
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU')

In [ ]:
# ── Cell 2: Install dependencies ──────────────────────────────
!pip install -q transformers==4.44.2 datasets==2.21.0 accelerate==0.33.0 \
    evaluate==0.4.2 rouge-score==0.1.2 sentencepiece==0.2.0 \
    scikit-learn==1.5.2 seaborn kaggle

In [ ]:
# ── Cell 3: Upload kaggle.json ────────────────────────────────
# This will show a file upload button
# Upload your kaggle.json from: https://www.kaggle.com/settings > API > Create New Token
from google.colab import files
import os

print('Upload your kaggle.json file...')
uploaded = files.upload()

os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('kaggle.json configured!')

In [ ]:
# ── Cell 4: Download dataset ──────────────────────────────────
import os
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('saved_models/sentiment', exist_ok=True)
os.makedirs('saved_models/summarization', exist_ok=True)

# downloading a different dataset that's always available
!kaggle datasets download -d snap/amazon-fine-food-reviews -p data/raw/ --unzip

import glob
files_found = glob.glob('data/raw/*')
print(f'Files in data/raw/: {files_found}')

In [ ]:
# ── Cell 5: Data pipeline ─────────────────────────────────────
import re, html, glob
import pandas as pd
from sklearn.model_selection import train_test_split

LABEL_MAP = {1: 0, 2: 0, 3: 1, 4: 2, 5: 2}

def clean_text(text):
    if not isinstance(text, str): return ''
    text = html.unescape(text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = pd.read_csv('data/raw/Reviews.csv', nrows=15000)
print(f'Loaded {len(df):,} rows')
print(f'Columns: {list(df.columns)}')

df = df.rename(columns={'Text': 'review_body', 'Score': 'star_rating', 'Summary': 'review_headline'})
df['review_body'] = df['review_body'].apply(clean_text)
df['review_headline'] = df['review_headline'].apply(clean_text)
df = df.dropna(subset=['review_body', 'star_rating'])
df = df[df['review_body'].str.len() > 20]
df = df.drop_duplicates(subset=['review_body'])
df['star_rating'] = pd.to_numeric(df['star_rating'], errors='coerce').astype(int)
df = df[df['star_rating'].between(1, 5)]
df['label'] = df['star_rating'].map(LABEL_MAP)

print(f'\nFinal: {len(df):,} rows')
print(df['label'].value_counts().sort_index())

cols = ['review_body', 'star_rating', 'label', 'review_headline']
train_df, temp = train_test_split(df[cols], test_size=0.30, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp, test_size=0.50, stratify=temp['label'], random_state=42)

train_df.to_csv('data/processed/train.csv', index=False)
val_df.to_csv('data/processed/val.csv', index=False)
test_df.to_csv('data/processed/test.csv', index=False)

print(f'\nTrain: {len(train_df):,}, Val: {len(val_df):,}, Test: {len(test_df):,}')
print('Data pipeline complete! ✅')

In [ ]:
# ── Cell 6: Train DistilBERT (Sentiment) ──────────────────────
# ~15 minutes on T4 GPU
import torch
import numpy as np
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from transformers import (
    DistilBertForSequenceClassification, DistilBertTokenizerFast,
    Trainer, TrainingArguments, DataCollatorWithPadding, EarlyStoppingCallback
)

ID2LABEL = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
LABEL2ID = {'Negative': 0, 'Neutral': 1, 'Positive': 2}

train_df = pd.read_csv('data/processed/train.csv')
val_df   = pd.read_csv('data/processed/val.csv')
test_df  = pd.read_csv('data/processed/test.csv')

tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def make_ds(df):
    ds = Dataset.from_pandas(df[['review_body','label']].rename(columns={'review_body':'text'}))
    return ds.map(lambda b: tokenizer(b['text'], truncation=True, max_length=256, padding=False),
                  batched=True, remove_columns=['text'])

train_ds = make_ds(train_df)
val_ds   = make_ds(val_df)
test_ds  = make_ds(test_df)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average='macro', zero_division=0)
    return {'accuracy': round(acc,4), 'f1_macro': round(f1,4),
            'precision': round(p,4), 'recall': round(r,4)}

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=3, id2label=ID2LABEL, label2id=LABEL2ID
)

args = TrainingArguments(
    output_dir='saved_models/sentiment/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    learning_rate=2e-5,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    fp16=True,
    logging_steps=50,
    save_total_limit=1,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=train_ds, eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('Training DistilBERT...')
trainer.train()

print('\nEvaluating on test set...')
results = trainer.evaluate(test_ds)
print(f'Test results: {results}')

trainer.save_model('saved_models/sentiment')
tokenizer.save_pretrained('saved_models/sentiment')
print('Sentiment model saved!')

In [ ]:
# ── Cell 7: Train T5-small (Simple training loop) ─────────────
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader, Dataset as TorchDataset
from transformers import T5ForConditionalGeneration, T5TokenizerFast
from torch.optim import AdamW

TASK_PREFIX = 'summarize: '
MAX_INPUT = 256
MAX_TARGET = 64
EPOCHS = 2
BATCH_SIZE = 8
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using: {device}')

train_df = pd.read_csv('data/processed/train.csv')
val_df   = pd.read_csv('data/processed/val.csv')

def get_target(row):
    if 'review_headline' in row and len(str(row.get('review_headline','')).strip()) > 5:
        return str(row['review_headline'])
    return str(row['review_body']).split('.')[0][:80]

train_df['target'] = train_df.apply(get_target, axis=1)
val_df['target']   = val_df.apply(get_target, axis=1)
train_df = train_df[train_df['review_body'].str.split().str.len() > 15].reset_index(drop=True)
val_df   = val_df[val_df['review_body'].str.split().str.len() > 15].reset_index(drop=True)

t5_tokenizer = T5TokenizerFast.from_pretrained('t5-small')
t5_model = T5ForConditionalGeneration.from_pretrained('t5-small').to(device)

class ReviewDataset(TorchDataset):
    def __init__(self, df):
        self.inputs  = [TASK_PREFIX + t for t in df['review_body'].tolist()]
        self.targets = df['target'].tolist()
    def __len__(self): return len(self.inputs)
    def __getitem__(self, i): return self.inputs[i], self.targets[i]

def collate_fn(batch):
    src, tgt = zip(*batch)
    enc = t5_tokenizer(list(src), max_length=MAX_INPUT, truncation=True, padding=True, return_tensors='pt')
    dec = t5_tokenizer(list(tgt), max_length=MAX_TARGET, truncation=True, padding=True, return_tensors='pt')
    labels = dec['input_ids'].clone()
    labels[labels == t5_tokenizer.pad_token_id] = -100
    return enc['input_ids'], enc['attention_mask'], labels

train_loader = DataLoader(ReviewDataset(train_df), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(ReviewDataset(val_df),   batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

optimizer = AdamW(t5_model.parameters(), lr=5e-5)

print('Training T5-small...')
for epoch in range(EPOCHS):
    t5_model.train()
    total_loss = 0
    for i, (input_ids, attention_mask, labels) in enumerate(train_loader):
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        loss = t5_model(input_ids=input_ids, attention_mask=attention_mask, labels=labels).loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if i % 100 == 0:
            print(f'  Epoch {epoch+1} | Step {i}/{len(train_loader)} | Loss: {loss.item():.4f}')
    print(f'Epoch {epoch+1} done. Avg loss: {total_loss/len(train_loader):.4f}')

t5_model.save_pretrained('saved_models/summarization')
t5_tokenizer.save_pretrained('saved_models/summarization')
print('Summarization model saved! ✅')

In [ ]:
# ── Cell 8: Package and download models ───────────────────────
# This zips both trained models so you can download them
import shutil

print('Packaging models...')
shutil.make_archive('reviewlens_models', 'zip', '.', 'saved_models')

import os
size_mb = os.path.getsize('reviewlens_models.zip') / (1024*1024)
print(f'ZIP size: {size_mb:.1f} MB')
print('Downloading...')

from google.colab import files
files.download('reviewlens_models.zip')
print('Done! Extract the ZIP and place the saved_models/ folder in your project root.')

## After downloading

1. Extract `reviewlens_models.zip`
2. You'll see a `saved_models/` folder with `sentiment/` and `summarization/` inside
3. Copy both folders into your project:
   ```
   reviewlens-ai/
   └── saved_models/
       ├── sentiment/      ← paste here
       └── summarization/  ← paste here
   ```
4. Restart your Flask app:
   ```powershell
   python app.py
   ```
5. Header should now show **models ready** in green!
   
---
*ReviewLens AI — Fine-tuned on Amazon Electronics Reviews*